<a href="https://colab.research.google.com/github/GODRATAN/FlyRank_ML-Intern/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GODRATAN/FlyRank_ML_Assignment/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Refreshing older pages

**Paper finding:** The FlyRank report states that, in its held-out analysis, 7 of 9 refresh strata showed statistically significant refresh lift. It reports a median effect for refreshed versus stale pages among 180+ day pages of +588 impressions, with a 95% confidence interval of [548, 634]. :chatgpt-content-reference{index="2"}

**Methodology question:** How was the refresh outcome/label defined, and how was the comparison group constructed? In particular, were refreshed and stale pages comparable before the refresh, and does the held-out validation design adequately separate the evidence used to define the comparison from the data used to measure the reported lift?

---

### Finding 2 — Future growth prediction

**Paper finding:** The report describes a growth-prediction model and reports performance on unseen pages from the same brands and on completely unseen brands, with different reported results across those settings. It also identifies variables such as days visible, days since update, content age, word count, and average position among the model's growth predictors. :chatgpt-content-reference{index="3"}

**Methodology question:** What exactly is the future-growth label and prediction window, and does the validation design prevent information from the future outcome window from entering the features? I would also want to confirm whether the unseen-brand evaluation is the primary estimate of generalization to new clients, rather than relying mainly on pages from already-seen brands.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Before vs after validation

**Before:** I use a random row split as a naive reference. This can place observations from the same client in both training and test sets.

**After:** I hold out whole clients using a grouped split. This prevents client-level overlap between training and test data and gives a stricter test of generalization to unseen clients.

Both evaluations use the same March 2026 feature window, the same April 2026 future-decline label, the same Logistic Regression method, and the same Precision@20 and Precision@50 metrics.

In [2]:
# ML-09 — Before vs after validation audit

import os
import numpy as np
import pandas as pd

from google.colab import userdata
from datasets import load_dataset

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Get the Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])


# --------------------------------------------------
# Same data design used in ML-08
# --------------------------------------------------

# March = feature window
# April = future outcome window

BASE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

needed_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]


def load_month(month):
    ds = load_dataset(
        "FlyRank/internship-warehouse",
        "fact_content_daily_performance",
        data_files={
            "train": f"{BASE}/month={month}/data_0.parquet"
        },
        token=HF_TOKEN,
    )["train"]

    return ds.select_columns(needed_cols).to_pandas()


march_raw = load_month("2026-03")
march_raw = march_raw[
    march_raw["gsc_data_available"] == True
].copy()

march_raw["gsc_avg_position"] = (
    march_raw["gsc_avg_position"].replace(0, np.nan)
)

march_agg = (
    march_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "mean"),
        march_days_observed=("report_date", "nunique"),
    )
)

del march_raw


april_raw = load_month("2026-04")
april_raw = april_raw[
    april_raw["gsc_data_available"] == True
].copy()

april_agg = (
    april_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_days_observed=("report_date", "nunique"),
    )
)

del april_raw


model_data = march_agg.merge(
    april_agg,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

model_data["march_daily_impressions"] = (
    model_data["march_impressions"]
    / model_data["march_days_observed"]
)

model_data["april_daily_impressions"] = (
    model_data["april_impressions"]
    / model_data["april_days_observed"]
)

eligible = (
    (model_data["march_impressions"] >= 100)
    & (model_data["march_days_observed"] >= 14)
    & (model_data["april_days_observed"] >= 14)
)

model_data = model_data[eligible].copy()

model_data["future_decline_label"] = (
    model_data["april_daily_impressions"]
    <= 0.80 * model_data["march_daily_impressions"]
).astype(int)

model_data["march_ctr"] = (
    model_data["march_clicks"]
    / model_data["march_impressions"].replace(0, np.nan)
)

feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_days_observed",
]


# --------------------------------------------------
# BEFORE — random row split
# --------------------------------------------------

X = model_data[feature_cols]
y = model_data["future_decline_label"]

X_train_before, X_test_before, y_train_before, y_test_before = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

before_model = make_model()
before_model.fit(X_train_before, y_train_before)

before_scores = before_model.predict_proba(
    X_test_before
)[:, 1]

before_p20 = precision_at_k(
    before_scores,
    y_test_before,
    20
)

before_p50 = precision_at_k(
    before_scores,
    y_test_before,
    50
)


# --------------------------------------------------
# AFTER — grouped by client
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_data,
        groups=model_data["client_hash_id"]
    )
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()

after_model = make_model()

after_model.fit(
    train_data[feature_cols],
    train_data["future_decline_label"]
)

after_scores = after_model.predict_proba(
    test_data[feature_cols]
)[:, 1]

after_p20 = precision_at_k(
    after_scores,
    test_data["future_decline_label"],
    20
)

after_p50 = precision_at_k(
    after_scores,
    test_data["future_decline_label"],
    50
)


# --------------------------------------------------
# Comparison table
# --------------------------------------------------

comparison = pd.DataFrame({
    "Validation": [
        "Before: random row split",
        "After: client-grouped split"
    ],
    "Precision@20": [
        before_p20,
        after_p20
    ],
    "Precision@50": [
        before_p50,
        after_p50
    ]
})

print(comparison.to_string(index=False))

print("\nBefore test rows:", len(X_test_before))
print("After test rows:", len(test_data))

print(
    "\nAfter client overlap:",
    len(
        set(train_data["client_hash_id"])
        & set(test_data["client_hash_id"])
    )
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

                 Validation  Precision@20  Precision@50
   Before: random row split          0.90          0.84
After: client-grouped split          0.85          0.82

Before test rows: 19254
After test rows: 22411

After client overlap: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# ML-09 — Leakage audit

# The final model features must come only from the March decision window.
# April is used only to create the future outcome label.

print("Final feature set:")
for feature in feature_cols:
    print("-", feature)

# 1. Check for obvious future/label-derived names.
forbidden_terms = [
    "future",
    "april",
    "label",
    "decline"
]

suspicious_features = [
    feature
    for feature in feature_cols
    if any(term in feature.lower() for term in forbidden_terms)
]

print("\nFeatures with suspicious future/label wording:")
print(suspicious_features if suspicious_features else "None")


# 2. Explicitly show the source window for every feature.
feature_sources = {
    "march_impressions": "March 2026",
    "march_clicks": "March 2026",
    "march_ctr": "March 2026",
    "march_avg_position": "March 2026",
    "march_days_observed": "March 2026",
}

print("\nFeature source audit:")
for feature in feature_cols:
    print(f"{feature}: {feature_sources.get(feature, 'UNKNOWN')}")


# 3. Confirm future outcome fields are not in the model features.
future_fields = [
    "april_impressions",
    "april_days_observed",
    "april_daily_impressions",
    "future_decline_label",
]

future_fields_used_as_features = [
    field
    for field in future_fields
    if field in feature_cols
]

print("\nFuture outcome fields used as features:")
print(
    future_fields_used_as_features
    if future_fields_used_as_features
    else "None"
)


# 4. Basic suspicious-correlation screen.
# This does not prove leakage; it only flags unusually strong relationships
# for manual review.
audit_corr = (
    model_data[feature_cols + ["future_decline_label"]]
    .corr(numeric_only=True)["future_decline_label"]
    .drop("future_decline_label")
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

print("\nFeature/label correlations:")
print(audit_corr)

Final feature set:
- march_impressions
- march_clicks
- march_ctr
- march_avg_position
- march_days_observed

Features with suspicious future/label wording:
None

Feature source audit:
march_impressions: March 2026
march_clicks: March 2026
march_ctr: March 2026
march_avg_position: March 2026
march_days_observed: March 2026

Future outcome fields used as features:
None

Feature/label correlations:
march_ctr             -0.149516
march_clicks          -0.084076
march_days_observed    0.058802
march_impressions     -0.047698
march_avg_position    -0.011247
Name: future_decline_label, dtype: float64


### Leakage audit conclusion

The final feature set contains only March 2026 decision-window variables:

- `march_impressions`
- `march_clicks`
- `march_ctr`
- `march_avg_position`
- `march_days_observed`

No April outcome fields or future-decline label fields are included as model features.

The feature-name audit found no future- or label-derived feature names. The correlation screen is treated only as a diagnostic and is not used by itself to determine whether leakage exists.

The future April window is used only to construct the outcome label, keeping the prediction inputs separate from the future outcome.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim:**

"Logistic Regression outperformed the Week-4 baseline."

**Safer claim:**

"On the held-out client test split used in this audit, Logistic Regression measured higher Precision@20 and Precision@50 than the Week-4 baseline: 0.85 vs 0.50 at @20 and 0.82 vs 0.52 at @50."

**Validation qualification:**

"Under the same model and feature setup, the measured precision was higher with a random row split (0.90 at @20; 0.84 at @50) than with the client-grouped split (0.85 at @20; 0.82 at @50). Therefore, the client-grouped result is the stricter estimate used for this audit."

**Interpretation:**

"These are measured results for this dataset, future-decline definition, and held-out-client split. They provide directional decision-support evidence and do not establish that Logistic Regression will generalize to all future clients or that the model causes better content outcomes."

## Self-check

Before you submit, confirm each line honestly:

- [YES] Every section above is filled — markdown thinking AND the code that backs it
- [YES] The notebook runs top to bottom with no errors (Runtime → Run all)
- [YES] No client names, URLs, or private queries anywhere
- [YES] My claims use careful words: observed, measured, directional, decision-support
- [YES] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.